In [1]:
# File aur folder operations ke liye os module import karta hai
import os

# CSV aur tabular data handle karne ke liye Pandas import karta hai
import pandas as pd

# Images open karne ke liye PIL import karta hai (size check me kaam aayega)
from PIL import Image

# Naye dataset folder ka path define karta hai (ab sab notebooks me yahi use hoga)
DATASET_DIR = r'C:\Projects\Internship_Learning\dataset_final'

# Images folder ka path banata hai
IMAGES_DIR = os.path.join(DATASET_DIR, 'images')

# Sabhi images ke labels wali CSV ka path define karta hai
LABELS_CSV = os.path.join(DATASET_DIR, 'labels.csv')

# Training set ki CSV ka path define karta hai
TRAIN_CSV = os.path.join(DATASET_DIR, 'train.csv')

# Validation set ki CSV ka path define karta hai
VAL_CSV = os.path.join(DATASET_DIR, 'val.csv')

# Test set ki CSV ka path define karta hai
TEST_CSV = os.path.join(DATASET_DIR, 'test.csv')

# Dataset folder exist karta hai ya nahi check karta hai
print("Dataset folder exists:", os.path.exists(DATASET_DIR))

# Images folder exist karta hai ya nahi check karta hai
print("Images folder exists :", os.path.exists(IMAGES_DIR))

# Images folder me sirf valid image files (png/jpg/jpeg) count karta hai
print("Total image files    :", len([f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]))

# Sabhi CSV files ko DataFrame me load karta hai
labels_df = pd.read_csv(LABELS_CSV)
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

# labels.csv ke column names print karta hai (verify karne ke liye)
print("\nlabels.csv columns:", list(labels_df.columns))

# Har CSV ki total rows print karta hai
print("labels:", len(labels_df), "| train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

# Poore dataset me har class (good/average/poor) ki count print karta hai
print("\nLabel distribution (all):")
print(labels_df['label'].value_counts())

# Poore dataset me har source (old/b1/b2/b3) ki count print karta hai
print("\nSource distribution (all):")
print(labels_df['source'].value_counts())

Dataset folder exists: True
Images folder exists : True
Total image files    : 979

labels.csv columns: ['filename', 'label', 'score', 'source', 'orig_filename']
labels: 979 | train: 685 | val: 147 | test: 147

Label distribution (all):
label
average    393
good       319
poor       267
Name: count, dtype: int64

Source distribution (all):
source
b3     337
b2     310
old    200
b1     132
Name: count, dtype: int64


In [2]:
# 1) CSV me jo filenames hain, woh disk pe actually maujood hain ya nahi check karta hai

# Images folder ki sabhi files ka set banata hai
disk_files = set(os.listdir(IMAGES_DIR))

# labels.csv ke sabhi filenames ka set banata hai
csv_files = set(labels_df['filename'])

# CSV me hai par disk pe nahi (missing images)
missing = [f for f in labels_df['filename'] if f not in disk_files]

# Disk pe hai par CSV me nahi (extra/unlabelled images)
extra = [f for f in disk_files if f not in csv_files and f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print("CSV me hai par disk pe nahi:", len(missing))
print("Disk pe hai par CSV me nahi :", len(extra))

# 2) labels.csv me duplicate filenames to nahi hain, yeh check karta hai
print("Duplicate filenames in labels.csv:", labels_df['filename'].duplicated().sum())

# 3) Data leakage check: train, val, test ke filenames ka overlap nikalta hai
s_tr = set(train_df['filename'])
s_va = set(val_df['filename'])
s_te = set(test_df['filename'])

print("\ntrain & val :", len(s_tr & s_va))
print("train & test:", len(s_tr & s_te))
print("val & test  :", len(s_va & s_te))

# Teeno splits ko jodne par poora labels.csv banta hai ya nahi check karta hai
print("Split total == labels total:", len(s_tr) + len(s_va) + len(s_te) == len(labels_df))

# Splits ke filenames labels.csv ke filenames se match karte hain ya nahi check karta hai
print("Split filenames == labels filenames:", (s_tr | s_va | s_te) == csv_files)

# 4) Har split me label aur source ka distribution print karta hai
for name, d in [('TRAIN', train_df), ('VAL', val_df), ('TEST', test_df)]:
    print(f"\n{name} ({len(d)})")
    print("Label :", d['label'].value_counts().to_dict())
    print("Source:", d['source'].value_counts().to_dict())

CSV me hai par disk pe nahi: 0
Disk pe hai par CSV me nahi : 0
Duplicate filenames in labels.csv: 0

train & val : 0
train & test: 0
val & test  : 0
Split total == labels total: True
Split filenames == labels filenames: True

TRAIN (685)
Label : {'average': 275, 'good': 223, 'poor': 187}
Source: {'b3': 236, 'b2': 216, 'old': 140, 'b1': 93}

VAL (147)
Label : {'average': 59, 'good': 48, 'poor': 40}
Source: {'b2': 52, 'b3': 45, 'old': 30, 'b1': 20}

TEST (147)
Label : {'average': 59, 'good': 48, 'poor': 40}
Source: {'b3': 56, 'b2': 42, 'old': 30, 'b1': 19}


In [3]:
# Image sizes ginne ke liye Counter import karta hai
from collections import Counter

# Har size (width, height) kitni images ka hai, uski count store karne ke liye
sizes = Counter()

# Har source (old/b1/b2/b3) ke sizes alag dekhne ke liye dictionary
sizes_by_source = {}

# Corrupt ya unreadable images store karne ke liye list
bad = []

# labels.csv ki har image ko ek-ek karke open karta hai
for _, row in labels_df.iterrows():
    try:
        # Image kholta hai (sirf header padhta hai, fast hai)
        with Image.open(os.path.join(IMAGES_DIR, row['filename'])) as im:
            # Overall size count update karta hai
            sizes[im.size] += 1
            # Source-wise size count update karta hai
            sizes_by_source.setdefault(row['source'], Counter())[im.size] += 1
    except Exception as e:
        # Agar image khul nahi payi to bad list me daal deta hai
        bad.append((row['filename'], str(e)))

# Corrupt images ki total count print karta hai
print("Corrupt/unreadable images:", len(bad))
for b in bad[:10]:
    print(b)

# Sabse common 8 sizes print karta hai
print("\nTop sizes (overall):")
for s, n in sizes.most_common(8):
    print(s, n)

# Har source ke top 3 sizes print karta hai
print("\nTop sizes (source-wise):")
for src, c in sizes_by_source.items():
    print(src, c.most_common(3))

# Final dataset summary print karta hai
print("\n" + "="*50)
print("DATASET SUMMARY (dataset_final)")
print("="*50)
print("Total   :", len(labels_df))
print("Good    :", (labels_df['label'] == 'good').sum())
print("Average :", (labels_df['label'] == 'average').sum())
print("Poor    :", (labels_df['label'] == 'poor').sum())
print("Train/Val/Test:", len(train_df), "/", len(val_df), "/", len(test_df))
print("="*50)

Corrupt/unreadable images: 0

Top sizes (overall):
(1900, 968) 782
(1899, 969) 7
(1900, 969) 5
(1899, 967) 4
(1897, 968) 4
(1898, 968) 4
(1898, 965) 3
(1898, 971) 3

Top sizes (source-wise):
old [((1899, 969), 7), ((1900, 969), 5), ((1899, 967), 4)]
b1 [((1900, 968), 132)]
b2 [((1900, 968), 310)]
b3 [((1900, 968), 337)]

DATASET SUMMARY (dataset_final)
Total   : 979
Good    : 319
Average : 393
Poor    : 267
Train/Val/Test: 685 / 147 / 147
